# ChromaDB Complete Guide (What it is, How it works, How to use it)

## 1. What is ChromaDB?

**ChromaDB (Chroma)** is an open-source **vector database / vector store** designed to help you build applications that need:
- **semantic search** (find similar meaning, not exact keywords),
- **Retrieval-Augmented Generation (RAG)** (retrieve relevant chunks, then ask an LLM),
- **embedding-based lookup** (store vectors and query by similarity).

At its core, Chroma stores:
- **documents** (your text chunks),
- **embeddings** (vectors for those chunks),
- **metadata** (extra fields like source, page, category),
- and provides **similarity search** + **metadata filtering**.

---

## 2. When should you use ChromaDB?

Use ChromaDB when you want:
- a simple way to store/query embeddings locally,
- a fast prototype for RAG,
- persistence on disk without running heavy infrastructure,
- metadata filtering like: `source="policy.pdf"` or `page=5`.

If you need multi-region scaling, strict HA, or massive multi-tenant production, you may later move to Pinecone/Milvus/Weaviate/Qdrant. But for learning, local apps, and many production workloads, Chroma can work well.

---

## 3. Key Concepts (must know)

### 3.1 Embeddings
Embeddings are numeric vectors that represent meaning. You generate them using an embedding model (OpenAI, Sentence Transformers, etc.).

Chroma itself does not “understand text”. It stores vectors and searches by distance/similarity.

### 3.2 Collections (VERY important)
A **collection** is like a “table” in a database, dedicated to a specific dataset.

A collection typically contains:
- `ids`: unique string IDs for each item
- `documents`: the raw text chunks
- `embeddings`: the vectors (optional if you use an embedding function)
- `metadatas`: key-value dictionaries
- `distances`: returned in results

**Rule of thumb**
- One app can have multiple collections.
- Example collections:
  - `company_policies`
  - `product_docs`
  - `support_tickets`

### 3.3 IDs
IDs are your primary keys. They must be unique within a collection.
Good pattern: `sourceName_page_chunkIndex`  
Example: `handbook_12_003`

### 3.4 Metadata
Metadata is how you filter results.
Example metadata:
- `{ "source": "handbook.pdf", "page": 12, "department": "HR" }`

Then query with filters like:
- “only from HR docs”
- “only page >= 10”
- “only source = handbook.pdf”

---

## 4. ChromaDB Deployment Modes

### Mode A: Embedded (local in your Python app)
This is the easiest. Great for notebooks, scripts, simple services.

### Mode B: Client/Server (Chroma server)
Chroma runs as a service, your app connects remotely.
Good when you want:
- separation of storage from app runtime
- multiple apps connecting to the same DB

This guide focuses on embedded mode because it’s the most common for starting.

---

## 5. Installing ChromaDB

Typical install:

- `pip install chromadb`

(If you are using LangChain, you may also install its extras depending on your setup.)

---

## 6. How to Create a ChromaDB (Persistent) and Use it Next Time

### 6.1 The most important detail: Persistence Directory
If you do not set persistence, Chroma may be **in-memory** (data disappears when the process ends).

So always use a folder like:
- `./chroma_db/`

That folder becomes your “database files”.

---

## 7. Example 1: Create (or Open) a Persistent ChromaDB + Collection 



In [1]:
import chromadb
from chromadb.config import Settings

PERSIST_DIR = "./chroma_db_test"

client = chromadb.Client(
    Settings(persist_directory=PERSIST_DIR)
)

# Create if not exists, otherwise reuse
collection = client.get_or_create_collection(name="my_collection")

print("Collections:", [c.name for c in client.list_collections()])

Collections: ['my_collection']


## 8. Example 2: Insert Data (documents + metadata)

In [2]:
docs = [
    "Vector databases store embeddings and allow similarity search.",
    "ChromaDB is a vector database used for semantic search and RAG."
]

metas = [
    {"topic": "vectors", "source": "notes"},
    {"topic": "chromadb", "source": "notes"}
]

ids = ["doc1", "doc2"]

collection.add(
    documents=docs,
    metadatas=metas,
    ids=ids
)


C:\Users\harry\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 59.9MiB/s]


## 9. Example 3: Query (Similarity Search)

In [3]:
#Query by text (if your setup supports query_texts)
results = collection.query(
    query_texts=["What is a vector database?"],
    n_results=2
)

print(results["documents"])
print(results["metadatas"])
print(results["distances"])


[['Vector databases store embeddings and allow similarity search.', 'ChromaDB is a vector database used for semantic search and RAG.']]
[[{'source': 'notes', 'topic': 'vectors'}, {'source': 'notes', 'topic': 'chromadb'}]]
[[0.621249258518219, 0.9293270707130432]]


In [6]:
# Query with metadata filtering
results = collection.query(
    query_texts=["What is ChromaDB?"],
    n_results=5,
    where={"topic": "chromadb"}
)
print(results["documents"])
print(results["metadatas"])

[['ChromaDB is a vector database used for semantic search and RAG.']]
[[{'topic': 'chromadb', 'source': 'notes'}]]


## 10. Example 4: Load the Same Collection Next Time (Reuse Existing DB)

In [7]:
import chromadb
from chromadb.config import Settings

PERSIST_DIR = "./chroma_db_test"

client = chromadb.Client(Settings(persist_directory=PERSIST_DIR))

# If you EXPECT it already exists, use get_collection
collection = client.get_collection(name="my_collection")

print("Count:", collection.count())


Count: 2


Important difference

get_or_create_collection("x") may create a new empty collection if you misspell the name.

get_collection("x") will fail if it doesn’t exist, which is safer for debugging.

## 11. CRUD Operations You Should Know

In [13]:
# Count records
print("Count:", collection.count())
# Get Itesms
items = collection.get(ids=["doc1", "doc2"])
print(items)

items = collection.get(include=["documents", "metadatas"])
print(items)

# Update items
collection.update(
    ids=["doc1"],
    documents=["A vector database is a specialized database for storing and querying vector embeddings."],
    metadatas=[{"topic": "vectors", "source": "updated_notes"}] 
)

# Delete items

collection.delete(ids=["doc2"])
print("Count after deletion:", collection.count())

Count: 2
{'ids': ['doc1', 'doc2'], 'embeddings': None, 'documents': ['A vector database is a specialized database for storing and querying vector embeddings.', 'ChromaDB is a vector database used for semantic search and RAG.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': 'updated_notes', 'topic': 'vectors'}, {'source': 'notes', 'topic': 'chromadb'}]}
{'ids': ['doc1', 'doc2'], 'embeddings': None, 'documents': ['A vector database is a specialized database for storing and querying vector embeddings.', 'ChromaDB is a vector database used for semantic search and RAG.'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [{'topic': 'vectors', 'source': 'updated_notes'}, {'source': 'notes', 'topic': 'chromadb'}]}
Count after deletion: 1


## 12. How ChromaDB Performs Similarity Search (Conceptually)

ChromaDB stores vector embeddings and retrieves results based on **vector similarity**, not exact text matching.

At a high level, the process is:
1. Your input text is converted into an embedding (vector).
2. Chroma compares this vector with stored vectors.
3. The closest vectors (by distance) are returned as results.

Common distance metrics used:
- **Cosine similarity** – most common for text embeddings
- **Dot product**
- **L2 (Euclidean distance)**

You usually do not manage the math directly. Chroma abstracts this and applies optimized search under the hood.

---

## 13. Best Practices (Strongly Recommended)

### 13.1 Use Proper Chunking

Avoid storing very large text blocks as a single embedding.

Recommended:
- Chunk size: **200–800 tokens**
- Overlap: **50–150 tokens**

This improves retrieval accuracy and context relevance.

---

### 13.2 Use Stable and Predictable IDs

Always use deterministic IDs so re-ingestion does not create duplicates.

Example pattern:
<source><page><chunk_index> 
employee_handbook_12_003


---

### 13.3 Store Useful Metadata

Metadata enables filtering and better control during retrieval.

Recommended metadata fields:
- `source` (filename or system name)
- `page` or `section`
- `doc_id`
- `category` or `domain`
- `timestamp` (optional)

This allows queries like:
- “Only HR documents”
- “Only page > 10”
- “Only policy documents”

---

### 13.4 Use the Same Embedding Model Consistently

If you store embeddings using one model, you **must** query using the same model.

Mixing embedding models leads to:
- Poor similarity scores
- Irrelevant or empty results

---

### 13.5 Separate Database Folder and Collection Names

Important distinction:
- **persist_directory** → where data is stored on disk
- **collection name** → logical dataset inside that directory

Multiple collections can exist inside the same persistence folder.

---

## 14. Common Mistakes and How to Fix Them

### Mistake 1: Data Disappears After Restart

**Cause**: Chroma was used without persistence.  
**Fix**: Always define a `persist_directory`.

---

### Mistake 2: Collection Appears Empty

**Causes**:
- Different folder path
- Misspelled collection name
- New collection accidentally created

**Fix**:
- Verify `persist_directory`
- Call `list_collections()`
- Use `get_collection()` when expecting existing data

---

### Mistake 3: Poor or Irrelevant Results

**Causes**:
- Bad chunking strategy
- Inconsistent embedding models
- No metadata filtering

**Fix**:
- Improve chunk size and overlap
- Use the same embedding model everywhere
- Add metadata and filters

---

## 15. Using ChromaDB with LangChain (RAG Setup)

ChromaDB integrates cleanly with LangChain for RAG pipelines.

Example:

```python
from langchain.vectorstores import Chroma
from langchain.embeddings.openai import OpenAIEmbeddings

PERSIST_DIR = "./chroma_db"
COLLECTION_NAME = "my_collection"

embeddings = OpenAIEmbeddings()

vectordb = Chroma(
    persist_directory=PERSIST_DIR,
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings
)

retriever = vectordb.as_retriever(search_kwargs={"k": 4})
docs = retriever.get_relevant_documents("Explain vector databases")


Important:

Use the same persist directory

Use the same collection name

Use the same embedding model

## 16. Pre-Usage Checklist (Before You Go Live)

Before using ChromaDB seriously, ensure:

Persistence directory is fixed and version-controlled (if needed)

Collection naming strategy is defined

Chunking strategy is finalized

Metadata schema is consistent

Embedding model choice is locked

IDs are stable and deterministic

You validated:

    list_collections()

    collection.count()

    Sample query quality

## 17. Final Summary

ChromaDB is best suited for local, lightweight, and fast vector search

Collections act like tables for embeddings

Persistence is critical for reuse

Metadata + chunking = high-quality retrieval

Start simple, design carefully, and scale later if needed